# Corrected checkpoint-selection comparison

This notebook reevaluates every saved LoRA checkpoint with the publication EOT/parser contract. It loads each BF16 base model once, swaps adapter checkpoints in memory, never loads merged/FP8/NVFP4 exports, preserves the old reports, and reports whether each selected checkpoint changed. Training is not run.

In [ ]:
# Cell 1: use the flat Drive code folder and prepare BF16 evaluation
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_BUNDLE_DIR = Path('/content/drive/MyDrive/colab_drive_bundle')
PACKAGE_ON_DRIVE = DRIVE_BUNDLE_DIR
WORKSPACE = Path('/content/checkpoint_selection_workspace')

if not (PACKAGE_ON_DRIVE / '__init__.py').is_file():
    raise FileNotFoundError(f'No package __init__.py found below {DRIVE_BUNDLE_DIR}')
shutil.rmtree(WORKSPACE, ignore_errors=True)
WORKSPACE.mkdir(parents=True)
# Copy top-level code only; never recursively copy model/result folders.
for source in PACKAGE_ON_DRIVE.iterdir():
    if source.is_file() and source.suffix in {'.py', '.sh', '.txt'}:
        shutil.copy2(source, WORKSPACE / source.name)
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))

try:
    token = userdata.get('HF_TOKEN') or userdata.get('hf_token')
except Exception:
    token = None
if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token

install_env = os.environ.copy()
install_env['INSTALL_LOW_PRECISION'] = '0'
subprocess.check_call(
    ['bash', str(WORKSPACE / 'install_colab.sh')],
    env=install_env,
)
# BF16 selection does not need TorchAO; stale Colab TorchAO breaks PEFT dispatch.
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
# Fail before downloading large models if adapter injection is still broken.
probe = '''import torch
from peft import LoraConfig, get_peft_model
model = torch.nn.Sequential(torch.nn.Linear(4, 4))
wrapped = get_peft_model(model, LoraConfig(r=2, target_modules=['0']))
print('BF16 PEFT adapter injection passed')
'''
subprocess.check_call([sys.executable, '-u', '-c', probe])
# Register this flat directory as a Python package for existing relative imports.
# This is an in-memory name, not a fen_move_colab folder on Drive or Colab.
for name in list(sys.modules):
    if name == 'fen_move_colab' or name.startswith('fen_move_colab.'):
        del sys.modules[name]
spec = importlib.util.spec_from_file_location('fen_move_colab', WORKSPACE / '__init__.py', submodule_search_locations=[str(WORKSPACE)])
package = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = package
spec.loader.exec_module(package)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])
print(f'Package source: {PACKAGE_ON_DRIVE}')


In [ ]:
# Cell 2: the six completed training runs
RUN_NAMES = [
    'gemma3_270m_all_legal_moves_v3',  # rank 32
    'gemma3_270m_all_legal_moves_v4',  # rank 16
    'gemma4_e2b_all_legal_moves_v3',   # rank 32
    'gemma4_e2b_all_legal_moves_v4',   # rank 16
    'gemma4_e4b_all_legal_moves_v3',   # rank 32
    'gemma4_e4b_all_legal_moves_v4',   # rank 16
]
VALIDATION_SAMPLES = 128
VALIDATION_BATCH_SIZE = 8
VALIDATION_MAX_NEW_TOKENS = 512

for run_name in RUN_NAMES:
    run_path = DRIVE_BUNDLE_DIR / 'runs' / run_name
    if not run_path.is_dir():
        raise FileNotFoundError(f'Missing configured run: {run_path}')
print(f'Ready to compare {len(RUN_NAMES)} runs. No merged or quantized exports will be loaded.')


In [ ]:
# Cell 3: run the resumable, shared-base comparison
# Load the evaluator from the flat workspace through its registered package name.
import importlib
compare_checkpoint_selections = importlib.import_module('fen_move_colab.compare_checkpoint_selections').compare_checkpoint_selections

SUMMARY = compare_checkpoint_selections(
    drive_bundle_dir=str(DRIVE_BUNDLE_DIR),
    run_names=RUN_NAMES,
    samples=VALIDATION_SAMPLES,
    batch_size=VALIDATION_BATCH_SIZE,
    max_new_tokens=VALIDATION_MAX_NEW_TOKENS,
)


In [ ]:
# Cell 4: compact decision table
import pandas as pd

display(pd.DataFrame(SUMMARY['results'])[[
    'run_name',
    'checkpoints_evaluated',
    'old_best_f1',
    'new_best_f1',
    'best_checkpoint_changed',
    'old_best_checkpoint',
    'new_best_checkpoint',
]])
print('Runs requiring re-export:', SUMMARY['changed_runs'])
